# Laboratori 3 · Ushtrime Shtesë — Merge Sort: Induksioni i Fortë dhe Pema Rekursive

> **Lidhja me leksionin:** *Leksioni 3 — Merge Sort* (slides 3–41).
>
> Laboratori kryesor e **mati** Merge Sort-in. Leksioni i përgjigjet dy pyetjeve me vërtetim:
> **A funksionon?** (induksion sipas gjatësisë së listës) dhe **A është i shpejtë?** (pema rekursive → $O(n\log n)$).
> Këtu i kthejmë të dyja vërtetimet në kod që mund ta ekzekutoni dhe ta shihni.
> Nuk kërkon Pygame.

## Kujtues teorik — Përça e sundo

```
MergeSort(A)
1   n = len(A)
2   if n ≤ 1:                       # lista me gjatësi ≤ 1
3       return A                    # është e renditur
4   L = MergeSort(A[0 : n/2])       # rendit gjysmën e majtë
5   R = MergeSort(A[n/2 : n])       # rendit gjysmën e djathtë
6   return Merge(L, R)              # kombino dy gjysmat e renditura
```

1. **PËRÇA** problemin në nënprobleme më të vogla të të njëjtit lloj;
2. **SUNDO** nënproblemet duke i zgjidhur rekursivisht;
3. **KOMBINO** zgjidhjet e nënproblemeve.

### Induksioni për algoritme iterative dhe rekursive (slide 27)

| | Algoritme iterative | Algoritme rekursive |
|---|---|---|
| **HI** | një kusht vlen pas iteracionit $i$ | algoritmi është korrekt për input me përmasë **të shumtën** $i$ |
| **Rasti bazë** | iteracioni 0 | $i$ < konstante e vogël (këtu: $i = 1$) |
| **Hapi induktiv** | HI për $k$ ⇒ HI për $k+1$ | HI për $\{1, \dots, k-1\}$ ⇒ HI për $k$ (**induksion i fortë**) |
| **Konkluzioni** | HI për numrin total të iteracioneve | HI për $i = n$ |

### Pema rekursive (slides 33–39)

| Niveli $t$ | # nënproblemeve | Gjatësia e secilit | Puna e `Merge` në nivel |
|:-:|:-:|:-:|:-:|
| 0 | 1 | $n$ | $O(n)$ |
| 1 | 2 | $n/2$ | $O(n)$ |
| $t$ | $2^t$ | $n/2^t$ | $O(n)$ |
| $\log n$ | $n$ | 1 | — |

$\log n + 1$ nivele × $O(n)$ punë për nivel $= O(n \log n)$.

In [1]:
# ── Mjeti i kontrollit ──────────────────────────────────────────────
# Ekzekutojeni këtë qelizë një herë, në fillim. Pas çdo ushtrimi, qeliza
# "kontrollo(...)" teston funksionin tuaj dhe ju tregon nëse është gati.

def kontrollo(titulli, testet):
    try:
        testet()
    except NotImplementedError:
        print(f"⏳ {titulli}: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.")
    except AssertionError as e:
        print(f"✗ {titulli}: {e}")
    except Exception as e:
        print(f"✗ {titulli}: kodi u ndal me gabim  {type(e).__name__}: {e}")
    else:
        print(f"✓ {titulli}: të gjitha testet kaluan!")

## Ushtrimi 1 — Logaritmi si numër përgjysmimesh (slide 31)

*Intuitivisht, $\log_2 n$ tregon sa herë duhet ta pjesëtoni $n$ me 2 për të arritur te 1.*

**Detyra:** plotësoni `numri_i_pergjysmimeve(n)` **pa përdorur** `math.log`: pjesëtoni me 2
(pjesëtim i plotë `//`) derisa të arrini 1 dhe numëroni hapat.

In [2]:
import math

def numri_i_pergjysmimeve(n):
    # PLOTESO: while n > 1: ...
    raise NotImplementedError


def _testet():
    for n, pritet in [(1, 0), (2, 1), (32, 5), (64, 6), (128, 7), (256, 8), (512, 9), (1000, 9)]:
        assert numri_i_pergjysmimeve(n) == pritet, f"numri_i_pergjysmimeve({n}) duhet të jetë {pritet}"
    for n in range(1, 5000):
        assert numri_i_pergjysmimeve(n) == math.floor(math.log2(n)), f"n={n}"

kontrollo("Ushtrimi 1 — Përgjysmimet", _testet)

⏳ Ushtrimi 1 — Përgjysmimet: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


In [3]:
print("log₂(numri i grimcave të universit ≈ 10⁸⁰) ≈", round(80 * math.log2(10), 1), "< 280   (slide 31)")
print()
print(f"{'n':>14} | {'log₂ n':>7} | {'n':>14} | {'n·log₂ n':>16} | {'n²':>24}")
for n in [16, 1_024, 1_000_000, 1_000_000_000]:
    print(f"{n:>14,} | {math.log2(n):>7.1f} | {n:>14,} | {round(n * math.log2(n)):>16,} | {n * n:>24,}")

log₂(numri i grimcave të universit ≈ 10⁸⁰) ≈ 265.8 < 280   (slide 31)

             n |  log₂ n |              n |         n·log₂ n |                       n²
            16 |     4.0 |             16 |               64 |                      256
         1,024 |    10.0 |          1,024 |           10,240 |                1,048,576
     1,000,000 |    19.9 |      1,000,000 |       19,931,569 |        1,000,000,000,000
 1,000,000,000 |    29.9 |  1,000,000,000 |   29,897,352,854 | 1,000,000,000,000,000,000


## Ushtrimi 2 — Pema rekursive, e matur nga kodi

Funksioni `merge_sort_regjistro` regjistron **çdo thirrje rekursive**: në cilin nivel të pemës ndodhi dhe sa
e gjatë ishte lista. Nga regjistri ndërtojmë tabelën e slide 38 — jo me teori, por me atë që ndodhi realisht.

In [4]:
from collections import defaultdict

def merge(L, R):
    S, i, j = [], 0, 0
    while i < len(L) and j < len(R):
        if L[i] <= R[j]: S.append(L[i]); i += 1
        else:            S.append(R[j]); j += 1
    return S + L[i:] + R[j:]


def merge_sort_regjistro(A, niveli=0, regjistri=None):
    if regjistri is None:
        regjistri = []
    n = len(A)
    regjistri.append((niveli, n))
    if n <= 1:
        return A, regjistri
    L, _ = merge_sort_regjistro(A[: n // 2], niveli + 1, regjistri)
    R, _ = merge_sort_regjistro(A[n // 2 :], niveli + 1, regjistri)
    return merge(L, R), regjistri


def tabela_e_pemes(regjistri):
    nivelet = defaultdict(list)
    for t, gjatesia in regjistri:
        nivelet[t].append(gjatesia)
    print(f"{'Niveli t':>9} | {'# nënprobleme':>14} | {'gjatësitë':>14} | {'puna e Merge':>13}")
    for t in sorted(nivelet):
        g = nivelet[t]
        puna = sum(x for x in g if x > 1)          # Merge bën punë vetëm kur lista ka > 1 element
        gjat = str(sorted(set(g))) if len(set(g)) > 1 else str(g[0])
        print(f"{t:>9} | {len(g):>14} | {gjat:>14} | {puna:>13}")


import random
_, reg = merge_sort_regjistro(random.sample(range(100), 16))
tabela_e_pemes(reg)

 Niveli t |  # nënprobleme |      gjatësitë |  puna e Merge
        0 |              1 |             16 |            16
        1 |              2 |              8 |            16
        2 |              4 |              4 |            16
        3 |              8 |              2 |            16
        4 |             16 |              1 |             0


**Vëzhgoni:** në çdo nivel që ka `Merge`, puna totale është **16 = n**. Ka $\log_2 16 + 1 = 5$ nivele.

**Detyra:**
1. `pema_teorike(n)` — për $n$ fuqi të 2-it, kthen listën e treshave `(t, numri_i_nenproblemeve, gjatesia)`
   për $t = 0, 1, \dots, \log_2 n$, sipas tabelës së leksionit.
2. `numri_i_niveleve(n)` — për **çdo** $n \ge 1$ (jo vetëm fuqi të 2-it), sa nivele ka pema?
   *Ndihmë: provoni `merge_sort_regjistro` për $n = 5, 6, 7, 9$ dhe shikoni nivelin maksimal.*

In [5]:
def pema_teorike(n):
    # PLOTESO: lista e (t, 2**t, n // 2**t)
    raise NotImplementedError


def numri_i_niveleve(n):
    # PLOTESO: ndihmë — math.ceil(math.log2(n)) + 1
    raise NotImplementedError


def _testet():
    for k in range(0, 11):
        n = 2 ** k
        _, reg = merge_sort_regjistro(list(range(n)))
        mates = defaultdict(list)
        for t, g in reg:
            mates[t].append(g)
        pritet = [(t, len(mates[t]), mates[t][0]) for t in sorted(mates)]
        assert pema_teorike(n) == pritet, f"n={n}: pema e matur është {pritet[:4]}…, ju ktheni {pema_teorike(n)[:4]}…"
    for n in range(1, 400):
        _, reg = merge_sort_regjistro(list(range(n)))
        nivele = max(t for t, _ in reg) + 1
        assert numri_i_niveleve(n) == nivele, f"n={n}: pema ka {nivele} nivele, ju ktheni {numri_i_niveleve(n)}"

kontrollo("Ushtrimi 2 — Pema rekursive", _testet)

⏳ Ushtrimi 2 — Pema rekursive: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


## Ushtrimi 3 — Induksioni i fortë në veprim

**HI (slide 26):** *Në çdo thirrje rekursive mbi një listë me gjatësi të shumtën $i$, MergeSort kthen një listë të renditur.*

Versioni më poshtë e kontrollon HI **në çdo kthim nga rekursioni**: rezultati duhet të jetë i renditur dhe
të ketë saktësisht të njëjtat elemente si inputi. Nëse të gjitha thirrjet për lista më të shkurtra janë
korrekte (HI), dhe `Merge` është korrekt, atëherë edhe thirrja aktuale është korrekte — ky është hapi induktiv.

**Detyra:** plotësoni `merge_sort(A)` (rekursiv, sipas pseudokodit, duke përdorur `merge` të dhënë më lart).

In [6]:
def merge_sort(A):
    # PLOTESO: rasti bazë + dy thirrje rekursive + merge
    raise NotImplementedError


thirrje = 0
def me_kontroll_HI(rendit):
    # Mbështjell një funksion renditjeje dhe kontrollon HI në çdo kthim.
    def i_mbeshtjelle(A):
        global thirrje
        thirrje += 1
        S = rendit(A)
        assert S == sorted(S) and sorted(S) == sorted(A), f"HI dështoi për inputin {A} → {S}"
        return S
    return i_mbeshtjelle


def _testet():
    global merge_sort, thirrje
    origjinali = merge_sort
    merge_sort = me_kontroll_HI(origjinali)      # edhe thirrjet rekursive kalojnë nga kontrolli
    try:
        from itertools import permutations
        for n in range(0, 8):
            for p in permutations(range(n)):
                merge_sort(list(p))
        rng = random.Random(3)
        for _ in range(200):
            L = [rng.randint(0, 50) for _ in range(rng.randint(0, 60))]
            assert merge_sort(L) == sorted(L)
    finally:
        merge_sort = origjinali
    print(f"   HI u kontrollua në {thirrje:,} thirrje rekursive — asnjë dështim.")

kontrollo("Ushtrimi 3 — merge_sort me induksion", _testet)

⏳ Ushtrimi 3 — merge_sort me induksion: ende pa plotësuar — zëvendësoni `raise NotImplementedError` me kodin tuaj.


### Kur dështon rasti bazë

Ky version ka rastin bazë `if n == 1` në vend të `if n <= 1`. Për çdo listë jo-bosh punon mirë.
Po për listën bosh?

In [7]:
import sys

def merge_sort_baze_e_gabuar(A):
    n = len(A)
    if n == 1:                                   # ← nuk mbulon n = 0
        return A
    L = merge_sort_baze_e_gabuar(A[: n // 2])
    R = merge_sort_baze_e_gabuar(A[n // 2 :])
    return merge(L, R)

print(merge_sort_baze_e_gabuar([5, 2, 9, 1]))
limit = sys.getrecursionlimit()
sys.setrecursionlimit(200)
try:
    merge_sort_baze_e_gabuar([])
except RecursionError:
    print("[] → RecursionError: rekursioni nuk ndalon kurrë!")
finally:
    sys.setrecursionlimit(limit)

[1, 2, 5, 9]
[] → RecursionError: rekursioni nuk ndalon kurrë!


> **Pyetje:** cili nga katër përbërësit e induksionit "dështon" këtu? Pse thirrja `merge_sort([])` nuk e
> zvogëlon problemin? *(Ndihmë: sa është `len(A[:0])`?)*

## Ushtrimi 4 — Teoria kundrejt matjes

Numërojmë krahasimet e `Merge Sort` dhe të `Insertion Sort` në lista të rastit dhe i krahasojmë me
$n \log_2 n$ dhe $n^2/4$. Raporti *krahasime / ($n\log_2 n$)* mbetet afërsisht **konstant** — kjo është
pikërisht ajo që thotë $O(n \log n)$.

In [8]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def krahasime_merge(A):
    c = 0
    def ms(A):
        nonlocal c
        if len(A) <= 1: return A
        L, R = ms(A[: len(A) // 2]), ms(A[len(A) // 2 :])
        S, i, j = [], 0, 0
        while i < len(L) and j < len(R):
            c += 1
            if L[i] <= R[j]: S.append(L[i]); i += 1
            else:            S.append(R[j]); j += 1
        return S + L[i:] + R[j:]
    ms(A)
    return c

def krahasime_insertion(A):
    A, c = list(A), 0
    for i in range(1, len(A)):
        aktual, j = A[i], i - 1
        while j >= 0:
            c += 1
            if A[j] <= aktual: break
            A[j + 1] = A[j]; j -= 1
        A[j + 1] = aktual
    return c

rng = random.Random(11)
ns = list(range(16, 1601, 64))
km, ki = [], []
for n in ns:
    L = rng.sample(range(10 * n), n)
    km.append(krahasime_merge(L)); ki.append(krahasime_insertion(L))

fig = make_subplots(rows=1, cols=2, subplot_titles=("Krahasimet", "Merge Sort: krahasime / (n·log₂n)"))
fig.add_scatter(x=ns, y=ki, mode="markers", name="Insertion Sort (matur)", row=1, col=1)
fig.add_scatter(x=ns, y=[n * n / 4 for n in ns], name="n²/4", line=dict(dash="dot"), row=1, col=1)
fig.add_scatter(x=ns, y=km, mode="markers", name="Merge Sort (matur)", row=1, col=1)
fig.add_scatter(x=ns, y=[n * math.log2(n) for n in ns], name="n·log₂n", line=dict(dash="dash"), row=1, col=1)
fig.add_scatter(x=ns, y=[k / (n * math.log2(n)) for k, n in zip(km, ns)], mode="lines+markers",
                name="raporti", row=1, col=2)
fig.update_yaxes(range=[0, 1.1], row=1, col=2)
fig.update_xaxes(title_text="n")
fig.update_layout(height=430, title="O(n²) kundrejt O(n log n) — matja konfirmon pemën rekursive")
fig.show()

## Punë në Klasë

**Problemi 1 — Pema rekursive me dorë.** Vizatoni pemën rekursive të `MergeSort` për
$A = [38, 27, 43, 3, 9, 82, 10, 15]$. Në çdo kulm shkruani listën; në rrugën e kthimit shkruani rezultatin e `Merge`.
Plotësoni tabelën:

| Niveli | # nënproblemeve | Gjatësia | Elementet e kombinuara (Merge) në këtë nivel |
|:-:|:-:|:-:|:-:|
| 0 | | | |
| 1 | | | |
| 2 | | | |
| 3 | | | — |

**Problemi 2 — Plotësoni hapin induktiv (slide 26: "Plotëso Hapin Induktiv!").**
Vërtetoni me induksion që `Merge(L, R)` është korrekt kur $L$ dhe $R$ janë të renditura.
- **HI:** pas $k$ hapave, $S[0:k]$ përmban $k$ elementet më të vegjël të $L \cup R$, të renditur; dhe $L[i]$, $R[j]$
  janë elementet më të vegjël që nuk janë vendosur ende.
- **Rasti bazë:** $k = 0$ — ………
- **Hapi induktiv:** supozojmë HI për $k$. Në hapin $k+1$ algoritmi zgjedh $\min(L[i], R[j])$. Pse ky është elementi
  i $(k+1)$-të më i vogël? ………
- **Konkluzioni:** ………

**Problemi 3 — Rekurrenca.** Le të jetë $T(n) = 2\,T(n/2) + n$ dhe $T(1) = 1$. Hapeni për $n = 8$:
$T(8) = 2T(4) + 8 = \dots$ Krahasoni rezultatin me $n \log_2 n + n$.

**Problemi 4.** Pse Merge Sort ka **të njëjtin** numër nivelesh për listën e renditur, për listën zbritëse dhe
për listën e rastit? Po Insertion Sort?

## Pyetje Reflektuese

1. Në vërtetimin e Insertion Sort-it, induksioni ishte sipas **numrit të iteracioneve**. Këtu është sipas
   **gjatësisë së listës**. Pse Merge Sort ka nevojë për induksion **të fortë** (HI për të gjitha gjatësitë $< k$)?
2. Leksioni supozon që $n$ është fuqi e 2-it. Çfarë ndryshon kur nuk është? (Shihni `numri_i_niveleve`.)
3. Merge Sort bën $O(n \log n)$ krahasime edhe për një listë **tashmë të renditur**. A mund ta përmirësoni
   që të jetë $O(n)$ në atë rast, pa e prishur rastin më të keq?
4. "A bëhet më mirë?" — a mendoni se ekziston algoritëm renditjeje me krahasime që është më i shpejtë se
   $O(n \log n)$ në rastin më të keq?